# 기호 회귀 실습

**Symbolic Regression**

데이터에 맞는 수식 자체를 탐색해 해석 가능한 관계식을 찾는 방법.

소재 분야에서 이해하기: 측정값에서 무차원 수 사이의 관계식을 찾는다.

이 노트북은 개념을 직접 돌려보기 위한 예제입니다. 데이터는 실제 측정값이 아니라 개념 확인용으로
생성한 값이므로, 결과 수치를 연구 결론으로 쓰지 마세요. 위에서부터 순서대로 실행하세요.
그림의 축 이름은 기본 폰트에 한글 글리프가 없어 영문으로 적었습니다.

참고 자료: [PySR 기호 회귀 문서](https://pysr.readthedocs.io/en/latest/)

## 1. 수식 자체를 탐색

후보 수식들을 만들어 데이터에 가장 맞는 식을 찾습니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
plt.rcParams['figure.figsize'] = (7, 4)

# 참 관계: y = 2.5 * x1 * sqrt(x2)  (관측 잡음 포함)
x1 = rng.uniform(1, 5, 200)
x2 = rng.uniform(1, 9, 200)
y_obs = 2.5 * x1 * np.sqrt(x2) + rng.normal(0, 0.3, 200)
print('관측 예: x1=%.2f x2=%.2f -> y=%.2f' % (x1[0], x2[0], y_obs[0]))

In [ ]:
import itertools
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

terms = {
    'x1': x1, 'x2': x2, 'x1*x2': x1 * x2, 'x1^2': x1 ** 2, 'sqrt(x2)': np.sqrt(x2),
    'x1*sqrt(x2)': x1 * np.sqrt(x2), 'log(x2)': np.log(x2), 'x1/x2': x1 / x2,
}
results = []
for size in (1, 2):
    for combination in itertools.combinations(terms, size):
        design = np.column_stack([terms[name] for name in combination])
        model = LinearRegression().fit(design, y_obs)
        results.append((r2_score(y_obs, model.predict(design)), size, combination, model))

results.sort(key=lambda row: (-row[0], row[1]))
for score, size, combination, model in results[:6]:
    formula = ' + '.join('%.2f*%s' % (c, n) for c, n in zip(model.coef_, combination))
    print('R2 %.4f  항 %d개  y = %s + %.2f' % (score, size, formula, model.intercept_))

## 2. 해석

가장 단순한 항 조합으로 가장 높은 설명력을 주는 식이 참 관계와 일치하는지 확인하세요.
실제 기호 회귀(PySR 등)는 훨씬 넓은 수식 공간을 유전 알고리즘으로 탐색하고, 복잡도에
벌점을 주어 해석 가능한 식을 남깁니다. 찾은 식이 인과관계를 뜻하지는 않습니다.

---

셀의 숫자를 바꿔가며 다시 실행해보면 개념이 더 분명해집니다. 용어 사전으로 돌아가려면
[소재·AI 용어 사전](https://forum.rnddata.org/glossary/#symbolic-regression)을 여세요.